# Search App-33 — Rush Hour : l'A* admissible sur un puzzle à glissement

**Navigation** : [<< App-34 15-puzzle A*](App-34-15Puzzle-Admissible-Astar-Python.ipynb) | [Index](../README.md) | [Version Python](App-33-RushHour-Admissible-Astar-Python.ipynb) | [Version C#](App-33-RushHour-Admissible-Astar-CSharp.ipynb)

Le **Rush Hour** est un puzzle à glissement de référence pour la recherche dans un espace d'états : une grille 6x6, une voiture rouge à extraire par la sortie en haut à droite, et une poignée de voitures qui se bloquent mutuellement. C'est le prolongement naturel du 15-puzzle, le puzzle à glissement de référence de la Partie 1 : comme lui, il offre un **espace d'états net**, un **test de but clair** et des **heuristiques dont la force se mesure** (états développés, longueur de solution).

Ce carnet pose le puzzle **from scratch** en Rust pur (std : `BinaryHeap`, `VecDeque`, `HashMap` — aucun crate externe) et y confronte trois algorithmes de recherche, sur un jeu d'instances à la difficulté contrôlable :

- **BFS** — optimale en nombre de coups, mais elle développe toute la frontière ;
- **A\*** — avec une **heuristique admissible** (les voitures qui bloquent la rouge) : optimal **et** nettement moins d'états développés ;
- **recherche gloutonne** — suit l'heuristique, rapide mais **sous-optimale** ;
- et un **benchmark comparatif** : sur un jeu de 6 instances distinctes à la difficulté comparable, le tableau **états développés x longueur de solution** par algorithme est le résultat mesuré du carnet.

**Pré-requis** : la Partie 1 de la série Search (espaces d'états, BFS/A* admissible — le 15-puzzle y est le terrain d'essai) et la Partie 2.
**Références** : IDA* (Korf 1985) ; le puzzle Rush Hour (Hider, 2003 — benchmarks `rush` / `rvect`) ; heuristiques admissibles pour les puzzles à glissement.

In [2]:
const N: usize = 6; // grille 6 x 6

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn car_cells(car: &Car) -> [(usize, usize); 2] {
    if car.o == Orientation::H {
        [(car.r, car.c), (car.r, car.c + 1)]
    } else {
        [(car.r, car.c), (car.r + 1, car.c)]
    }
}

fn board_occupied(cars: &[Car]) -> std::collections::HashSet<(usize, usize)> {
    let mut occ = std::collections::HashSet::new();
    for car in cars {
        for &cell in &car_cells(car) {
            occ.insert(cell);
        }
    }
    occ
}

fn red_car(cars: &[Car]) -> Option<Car> {
    for car in cars {
        if car.red {
            return Some(*car);
        }
    }
    None
}

println!("Primitives chargees : grille {} x {}", N, N);

Primitives chargees : grille 6 x 6


## 1. L'espace d'états : états, déplacements, test de but

Un **état** est la position de toutes les voitures. Chaque voiture est une struct `Car { r, c, o, red }` : pour une voiture horizontale, `(r, c)` est sa cellule de gauche (elle occupe `(r,c)` et `(r,c+1)`) ; pour une verticale, c'est sa cellule du haut.

Un **déplacement** glisse une voiture le long de son axe vers toutes les positions vides atteignables : faire glisser une voiture, quelle que soit la distance, compte pour **un coup** (c'est la règle du puzzle). Le **but** est atteint quand la voiture rouge occupe les deux cases du haut à droite, `(0,4)` et `(0,5)`.

In [3]:
const N: usize = 6;

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn car_cells(car: &Car) -> [(usize, usize); 2] {
    if car.o == Orientation::H {
        [(car.r, car.c), (car.r, car.c + 1)]
    } else {
        [(car.r, car.c), (car.r + 1, car.c)]
    }
}

fn board_occupied(cars: &[Car]) -> std::collections::HashSet<(usize, usize)> {
    let mut occ = std::collections::HashSet::new();
    for car in cars {
        for &cell in &car_cells(car) {
            occ.insert(cell);
        }
    }
    occ
}

fn red_car(cars: &[Car]) -> Option<Car> {
    for car in cars {
        if car.red {
            return Some(*car);
        }
    }
    None
}

fn state_key(cars: &[Car]) -> Vec<Car> {
    let mut v = cars.to_vec();
    v.sort();
    v
}

fn is_goal(cars: &[Car]) -> bool {
    match red_car(cars) {
        Some(r) => r.r == 0 && r.c == 4 && r.o == Orientation::H,
        None => false,
    }
}

fn slide_positions(car: &Car, occ: &std::collections::HashSet<(usize, usize)>) -> Vec<Car> {
    let mut pos = Vec::new();
    if car.o == Orientation::H {
        let mut k = 1;
        while car.c + k + 1 < N && !occ.contains(&(car.r, car.c + k + 1)) {
            pos.push(Car { r: car.r, c: car.c + k, o: Orientation::H, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.c >= k && !occ.contains(&(car.r, car.c - k)) {
            pos.push(Car { r: car.r, c: car.c - k, o: Orientation::H, red: car.red });
            k += 1;
        }
    } else { // V
        let mut k = 1;
        while car.r + k + 1 < N && !occ.contains(&(car.r + k + 1, car.c)) {
            pos.push(Car { r: car.r + k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.r >= k && !occ.contains(&(car.r - k, car.c)) {
            pos.push(Car { r: car.r - k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
    }
    pos
}

fn neighbors(cars: &[Car]) -> Vec<Vec<Car>> {
    let occ = board_occupied(cars);
    let mut out = std::collections::BTreeSet::new();
    for (i, car) in cars.iter().enumerate() {
        for newpos in slide_positions(car, &occ) {
            if newpos != *car {
                let mut newcars = cars.to_vec();
                newcars[i] = newpos;
                out.insert(state_key(&newcars));
            }
        }
    }
    out.into_iter().collect() // tri -> ordre deterministe (BFS/A*/glouton reproductibles)
}

fn show_board(cars: &[Car]) -> String {
    let mut occ: std::collections::HashMap<(usize, usize), char> = std::collections::HashMap::new();
    for car in cars {
        for &cell in &car_cells(car) {
            let ch = if car.red { 'R' } else if car.o == Orientation::H { '-' } else { '|' };
            occ.insert(cell, ch);
        }
    }
    let mut lines = Vec::new();
    for i in 0..N {
        let mut parts: Vec<String> = Vec::new();
        for j in 0..N {
            parts.push(occ.get(&(i, j)).copied().unwrap_or('.').to_string());
        }
        let mut line = parts.join("  ");
        if i == 0 {
            line.push_str("   <== SORTIE");
        }
        lines.push(line);
    }
    lines.join("\n")
}

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
fn py_car(c: &Car) -> String {
    let os = if c.o == Orientation::H { "H" } else { "V" };
    format!("({}, {}, '{}', {})", c.r, c.c, os, py_bool(c.red))
}
fn py_key(k: &[Car]) -> String {
    format!("({})", k.iter().map(|c| py_car(c)).collect::<Vec<_>>().join(", "))
}

// un etat de depart simple pour la demonstration
let demo = vec![
    Car { r: 0, c: 2, o: Orientation::H, red: true },
    Car { r: 2, c: 0, o: Orientation::V, red: false },
    Car { r: 4, c: 3, o: Orientation::H, red: false },
];
println!("{}", show_board(&demo));
println!("voitures: {} | etat cle: {}", demo.len(), py_key(&state_key(&demo)));

.  .  R  R  .  .   <== SORTIE


.  .  .  .  .  .


|  .  .  .  .  .


|  .  .  .  .  .


.  .  .  -  -  .


.  .  .  .  .  .


voitures: 3 | etat cle: ((0, 2, 'H', True), (2, 0, 'V', False), (4, 3, 'H', False))


La grille est lue : la voiture rouge (`R`) est en haut, la sortie est à droite, et les autres voitures (`-` horizontales, `|` verticales) tiennent la grille. `neighbors` énumère tous les glissements légaux en un coup ; `is_goal` teste si la rouge est à la sortie.

## 2. Instances : une position de base + génération par scramble

Plutôt que de chercher à la main une position résolable, on **génère** des instances **garanties résolables** : on part d'une **position de base** (un enchevêtrement donné, résolable en 3 coups) et on lui applique quelques glissements **alatoires** — l'inverse de cette suite est, par construction, un chemin vers le but. C'est exactement la méthode par laquelle on contrôle la difficulté d'un puzzle : la position de base porte le caractère « dur », le scramble ne fait que décliner des variantes distinctes et résolables.

In [4]:
const N: usize = 6;

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn car_cells(car: &Car) -> [(usize, usize); 2] {
    if car.o == Orientation::H {
        [(car.r, car.c), (car.r, car.c + 1)]
    } else {
        [(car.r, car.c), (car.r + 1, car.c)]
    }
}

fn board_occupied(cars: &[Car]) -> std::collections::HashSet<(usize, usize)> {
    let mut occ = std::collections::HashSet::new();
    for car in cars {
        for &cell in &car_cells(car) {
            occ.insert(cell);
        }
    }
    occ
}

fn red_car(cars: &[Car]) -> Option<Car> {
    for car in cars {
        if car.red {
            return Some(*car);
        }
    }
    None
}

fn state_key(cars: &[Car]) -> Vec<Car> {
    let mut v = cars.to_vec();
    v.sort();
    v
}

fn is_goal(cars: &[Car]) -> bool {
    match red_car(cars) {
        Some(r) => r.r == 0 && r.c == 4 && r.o == Orientation::H,
        None => false,
    }
}

fn slide_positions(car: &Car, occ: &std::collections::HashSet<(usize, usize)>) -> Vec<Car> {
    let mut pos = Vec::new();
    if car.o == Orientation::H {
        let mut k = 1;
        while car.c + k + 1 < N && !occ.contains(&(car.r, car.c + k + 1)) {
            pos.push(Car { r: car.r, c: car.c + k, o: Orientation::H, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.c >= k && !occ.contains(&(car.r, car.c - k)) {
            pos.push(Car { r: car.r, c: car.c - k, o: Orientation::H, red: car.red });
            k += 1;
        }
    } else {
        let mut k = 1;
        while car.r + k + 1 < N && !occ.contains(&(car.r + k + 1, car.c)) {
            pos.push(Car { r: car.r + k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.r >= k && !occ.contains(&(car.r - k, car.c)) {
            pos.push(Car { r: car.r - k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
    }
    pos
}

fn neighbors(cars: &[Car]) -> Vec<Vec<Car>> {
    let occ = board_occupied(cars);
    let mut out = std::collections::BTreeSet::new();
    for (i, car) in cars.iter().enumerate() {
        for newpos in slide_positions(car, &occ) {
            if newpos != *car {
                let mut newcars = cars.to_vec();
                newcars[i] = newpos;
                out.insert(state_key(&newcars));
            }
        }
    }
    out.into_iter().collect()
}

fn show_board(cars: &[Car]) -> String {
    let mut occ: std::collections::HashMap<(usize, usize), char> = std::collections::HashMap::new();
    for car in cars {
        for &cell in &car_cells(car) {
            let ch = if car.red { 'R' } else if car.o == Orientation::H { '-' } else { '|' };
            occ.insert(cell, ch);
        }
    }
    let mut lines = Vec::new();
    for i in 0..N {
        let mut parts: Vec<String> = Vec::new();
        for j in 0..N {
            parts.push(occ.get(&(i, j)).copied().unwrap_or('.').to_string());
        }
        let mut line = parts.join("  ");
        if i == 0 {
            line.push_str("   <== SORTIE");
        }
        lines.push(line);
    }
    lines.join("\n")
}

// MT19937 — déterministe, compatible random.Random de CPython (init_by_array)
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn base_hard() -> [Car; 6] {
    [
        Car { r: 0, c: 0, o: Orientation::H, red: true },   // rouge, loin de la sortie
        Car { r: 0, c: 2, o: Orientation::V, red: false },  // bloque la ligne 0, col 2
        Car { r: 0, c: 4, o: Orientation::V, red: false },  // bloque la ligne 0, col 4
        Car { r: 3, c: 1, o: Orientation::H, red: false },
        Car { r: 4, c: 3, o: Orientation::H, red: false },
        Car { r: 2, c: 5, o: Orientation::V, red: false },
    ]
}

fn scramble(cars: &[Car], steps: usize, seed: u32) -> Vec<Car> {
    let mut rng = Mt19937::new_by_array(&[seed]);
    let mut cur = state_key(cars);
    for _ in 0..steps {
        let nb = neighbors(&cur);
        cur = rng.choice(&nb).clone();
    }
    cur
}

let instance = base_hard();
println!("Position de depart (position de base, resolable en 3 coups) :");
println!("{}", show_board(&instance));
println!("nb de voitures: {}", instance.len());

Position de depart (position de base, resolable en 3 coups) :


R  R  |  .  |  .   <== SORTIE


.  .  |  .  |  .


.  .  .  .  .  |


.  -  -  .  .  |


.  .  .  -  -  .


.  .  .  .  .  .


nb de voitures: 6


La position de départ est lue : la rouge est en haut à gauche, éloignée de la sortie, et deux voitures verticales (`|`) lui barrent la ligne 0 en colonnes 2 et 4 — elle doit d'abord les écarter. Cette instance est **résolable** (en 3 coups, comme on va le vérifier en la résolant) ; le scramble générera ensuite des variantes distinctes de même difficulté pour le benchmark.

## 3. BFS — l'optimum en nombre de coups (baseline)

Le **BFS** explore l'espace par niveaux : il garantit de trouver une solution **optimale** (le plus petit nombre de coups) car il atteint chaque état par le chemin le plus court. Son prix : il développe l'intégralité de la frontière explorée, sans aucune information sur la direction du but. C'est notre **baseline** de référence.

In [5]:
const N: usize = 6;

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn car_cells(car: &Car) -> [(usize, usize); 2] {
    if car.o == Orientation::H {
        [(car.r, car.c), (car.r, car.c + 1)]
    } else {
        [(car.r, car.c), (car.r + 1, car.c)]
    }
}

fn board_occupied(cars: &[Car]) -> std::collections::HashSet<(usize, usize)> {
    let mut occ = std::collections::HashSet::new();
    for car in cars {
        for &cell in &car_cells(car) {
            occ.insert(cell);
        }
    }
    occ
}

fn red_car(cars: &[Car]) -> Option<Car> {
    for car in cars {
        if car.red { return Some(*car); }
    }
    None
}

fn state_key(cars: &[Car]) -> Vec<Car> {
    let mut v = cars.to_vec();
    v.sort();
    v
}

fn is_goal(cars: &[Car]) -> bool {
    match red_car(cars) {
        Some(r) => r.r == 0 && r.c == 4 && r.o == Orientation::H,
        None => false,
    }
}

fn slide_positions(car: &Car, occ: &std::collections::HashSet<(usize, usize)>) -> Vec<Car> {
    let mut pos = Vec::new();
    if car.o == Orientation::H {
        let mut k = 1;
        while car.c + k + 1 < N && !occ.contains(&(car.r, car.c + k + 1)) {
            pos.push(Car { r: car.r, c: car.c + k, o: Orientation::H, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.c >= k && !occ.contains(&(car.r, car.c - k)) {
            pos.push(Car { r: car.r, c: car.c - k, o: Orientation::H, red: car.red });
            k += 1;
        }
    } else {
        let mut k = 1;
        while car.r + k + 1 < N && !occ.contains(&(car.r + k + 1, car.c)) {
            pos.push(Car { r: car.r + k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.r >= k && !occ.contains(&(car.r - k, car.c)) {
            pos.push(Car { r: car.r - k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
    }
    pos
}

fn neighbors(cars: &[Car]) -> Vec<Vec<Car>> {
    let occ = board_occupied(cars);
    let mut out = std::collections::BTreeSet::new();
    for (i, car) in cars.iter().enumerate() {
        for newpos in slide_positions(car, &occ) {
            if newpos != *car {
                let mut newcars = cars.to_vec();
                newcars[i] = newpos;
                out.insert(state_key(&newcars));
            }
        }
    }
    out.into_iter().collect()
}

fn base_hard() -> [Car; 6] {
    [
        Car { r: 0, c: 0, o: Orientation::H, red: true },
        Car { r: 0, c: 2, o: Orientation::V, red: false },
        Car { r: 0, c: 4, o: Orientation::V, red: false },
        Car { r: 3, c: 1, o: Orientation::H, red: false },
        Car { r: 4, c: 3, o: Orientation::H, red: false },
        Car { r: 2, c: 5, o: Orientation::V, red: false },
    ]
}

// Recherche en largeur : solution optimale (min de coups), developpe tout le frontiere.
fn bfs(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let start = state_key(start_cars);
    let mut q: std::collections::VecDeque<Vec<Car>> = std::collections::VecDeque::new();
    q.push_back(start.clone());
    let mut came_from: std::collections::HashMap<Vec<Car>, Option<Vec<Car>>> = std::collections::HashMap::new();
    came_from.insert(start, None);
    let mut nodes = 0;
    while let Some(state) = q.pop_front() {
        nodes += 1;
        if is_goal(&state) {
            let mut path = vec![state.clone()];
            let mut cur = state;
            loop {
                match came_from.get(&cur) {
                    Some(Some(parent)) => { path.push(parent.clone()); cur = parent.clone(); }
                    _ => break,
                }
            }
            path.reverse();
            return (Some(path), nodes);
        }
        for nb in neighbors(&state) {
            if !came_from.contains_key(&nb) {
                came_from.insert(nb.clone(), Some(state.clone()));
                q.push_back(nb);
            }
        }
    }
    (None, nodes)
}

let instance = base_hard();
let (path, nodes) = bfs(&instance);
let path = path.unwrap();
println!("BFS : solution en {} coups, {} etats developpes", path.len() - 1, nodes);

BFS : solution en 3 coups, 433 etats developpes


Le BFS retrouve une solution **optimale** mais en développant un nombre d'états bien supérieur au nécessaire : il ne sait pas où est le but, il balaye toute la frontière. C'est exactement le défaut qu'une heuristique admissible doit corriger.

## 4. A* avec heuristique admissible

L'**heuristique** que nous posons est simple et **admissible** : elle compte le nombre de voitures (hors la rouge) qui occupent la **ligne 0 à droite de la voiture rouge**. Pourquoi admissible ? Pour que la rouge atteigne la sortie, **chaque** une de ces voitures doit être déplacée **au moins une fois** hors de sa trajectoire. Le nombre de ces voitures est donc une **borne inférieure** au nombre de coups restants — l'A* qui l'utilise reste **optimal** (il ne coupe jamais une solution meilleure), tout en focalisant l'exploration vers le but.

In [6]:
const N: usize = 6;

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn car_cells(car: &Car) -> [(usize, usize); 2] {
    if car.o == Orientation::H {
        [(car.r, car.c), (car.r, car.c + 1)]
    } else {
        [(car.r, car.c), (car.r + 1, car.c)]
    }
}

fn board_occupied(cars: &[Car]) -> std::collections::HashSet<(usize, usize)> {
    let mut occ = std::collections::HashSet::new();
    for car in cars {
        for &cell in &car_cells(car) {
            occ.insert(cell);
        }
    }
    occ
}

fn red_car(cars: &[Car]) -> Option<Car> {
    for car in cars {
        if car.red { return Some(*car); }
    }
    None
}

fn state_key(cars: &[Car]) -> Vec<Car> {
    let mut v = cars.to_vec();
    v.sort();
    v
}

fn is_goal(cars: &[Car]) -> bool {
    match red_car(cars) {
        Some(r) => r.r == 0 && r.c == 4 && r.o == Orientation::H,
        None => false,
    }
}

fn slide_positions(car: &Car, occ: &std::collections::HashSet<(usize, usize)>) -> Vec<Car> {
    let mut pos = Vec::new();
    if car.o == Orientation::H {
        let mut k = 1;
        while car.c + k + 1 < N && !occ.contains(&(car.r, car.c + k + 1)) {
            pos.push(Car { r: car.r, c: car.c + k, o: Orientation::H, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.c >= k && !occ.contains(&(car.r, car.c - k)) {
            pos.push(Car { r: car.r, c: car.c - k, o: Orientation::H, red: car.red });
            k += 1;
        }
    } else {
        let mut k = 1;
        while car.r + k + 1 < N && !occ.contains(&(car.r + k + 1, car.c)) {
            pos.push(Car { r: car.r + k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.r >= k && !occ.contains(&(car.r - k, car.c)) {
            pos.push(Car { r: car.r - k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
    }
    pos
}

fn neighbors(cars: &[Car]) -> Vec<Vec<Car>> {
    let occ = board_occupied(cars);
    let mut out = std::collections::BTreeSet::new();
    for (i, car) in cars.iter().enumerate() {
        for newpos in slide_positions(car, &occ) {
            if newpos != *car {
                let mut newcars = cars.to_vec();
                newcars[i] = newpos;
                out.insert(state_key(&newcars));
            }
        }
    }
    out.into_iter().collect()
}

fn base_hard() -> [Car; 6] {
    [
        Car { r: 0, c: 0, o: Orientation::H, red: true },
        Car { r: 0, c: 2, o: Orientation::V, red: false },
        Car { r: 0, c: 4, o: Orientation::V, red: false },
        Car { r: 3, c: 1, o: Orientation::H, red: false },
        Car { r: 4, c: 3, o: Orientation::H, red: false },
        Car { r: 2, c: 5, o: Orientation::V, red: false },
    ]
}

fn bfs(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let start = state_key(start_cars);
    let mut q: std::collections::VecDeque<Vec<Car>> = std::collections::VecDeque::new();
    q.push_back(start.clone());
    let mut came_from: std::collections::HashMap<Vec<Car>, Option<Vec<Car>>> = std::collections::HashMap::new();
    came_from.insert(start, None);
    let mut nodes = 0;
    while let Some(state) = q.pop_front() {
        nodes += 1;
        if is_goal(&state) {
            let mut path = vec![state.clone()];
            let mut cur = state;
            loop {
                match came_from.get(&cur) {
                    Some(Some(parent)) => { path.push(parent.clone()); cur = parent.clone(); }
                    _ => break,
                }
            }
            path.reverse();
            return (Some(path), nodes);
        }
        for nb in neighbors(&state) {
            if !came_from.contains_key(&nb) {
                came_from.insert(nb.clone(), Some(state.clone()));
                q.push_back(nb);
            }
        }
    }
    (None, nodes)
}

// Voitures (hors rouge) bloquant la ligne 0 a droite de la rouge -> borne admissible.
fn heuristic(cars: &[Car]) -> usize {
    let r = match red_car(cars) { Some(r) => r, None => return 0 };
    if r.r != 0 { return 0; }
    let red_right = r.c + 1; // case droite de la rouge
    let _occ = board_occupied(cars);
    let mut blockers: std::collections::HashSet<(usize, usize, Orientation, bool)> = std::collections::HashSet::new();
    for car in cars {
        if car.red { continue; }
        for &(a, b) in &car_cells(car) {
            if a == 0 && b > red_right {
                blockers.insert((car.r, car.c, car.o, car.red));
                break;
            }
        }
    }
    blockers.len()
}

#[derive(Clone, PartialEq, Eq, PartialOrd, Ord)]
struct HeapKey { f: usize, g: usize, key: Vec<Car> }

// A* : optimal (heuristique admissible) et cibble l'exploration vers le but.
fn astar(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let start = state_key(start_cars);
    let mut heap: std::collections::BinaryHeap<std::cmp::Reverse<HeapKey>> = std::collections::BinaryHeap::new();
    heap.push(std::cmp::Reverse(HeapKey { f: heuristic(&start), g: 0, key: start.clone() }));
    let mut g_score: std::collections::HashMap<Vec<Car>, usize> = std::collections::HashMap::new();
    g_score.insert(start.clone(), 0);
    let mut came_from: std::collections::HashMap<Vec<Car>, Vec<Car>> = std::collections::HashMap::new();
    let mut closed: std::collections::HashSet<Vec<Car>> = std::collections::HashSet::new();
    let mut nodes = 0;
    while let Some(std::cmp::Reverse(hk)) = heap.pop() {
        if closed.contains(&hk.key) { continue; }
        closed.insert(hk.key.clone());
        nodes += 1;
        if is_goal(&hk.key) {
            let mut path = vec![hk.key.clone()];
            let mut cur = hk.key;
            while let Some(parent) = came_from.get(&cur) {
                path.push(parent.clone());
                cur = parent.clone();
            }
            path.reverse();
            return (Some(path), nodes);
        }
        for nb in neighbors(&hk.key) {
            let ng = hk.g + 1;
            let inf = usize::MAX;
            if ng < *g_score.get(&nb).unwrap_or(&inf) {
                g_score.insert(nb.clone(), ng);
                came_from.insert(nb.clone(), hk.key.clone());
                heap.push(std::cmp::Reverse(HeapKey { f: ng + heuristic(&nb), g: ng, key: nb }));
            }
        }
    }
    (None, nodes)
}

let instance = base_hard();
let (path_a, nodes_a) = astar(&instance);
let path_a = path_a.unwrap();
// recalcule bfs pour le rapport (cellule autonome sous evcxr)
let (_, nodes_bfs) = bfs(&instance);
println!("A*    : solution en {} coups, {} etats developpes", path_a.len() - 1, nodes_a);
println!("rapport BFS/A* en etats developpes : {:.1}x", nodes_bfs as f64 / nodes_a as f64);

A*    : solution en 3 coups, 100 etats developpes


rapport BFS/A* en etats developpes : 4.3x


L'A* retrouve **la même longueur de solution optimale** que le BFS (l'heuristique est admissible, l'optimalité est préservée) en développant **nettement moins d'états** : c'est tout le rendement d'une heuristique admissible. Le rapport développé ci-dessus est le chiffre central de la comparaison.

## 5. Recherche gloutonne — rapide, mais sous-optimale

La **recherche gloutonne** ne garde aucun compte du coût déjà parcouru : à chaque étape elle part vers le voisin dont l'heuristique est la **plus faible** (le moins de bloqueurs). Plus simple que l'A* (elle ne s'embarrasse pas de l'optimalité), elle **abandonne la garantie d'optimalité** : elle peut livrer une solution nettement plus longue, voire s'engager dans un cul-de-sac.

In [7]:
const N: usize = 6;

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn car_cells(car: &Car) -> [(usize, usize); 2] {
    if car.o == Orientation::H {
        [(car.r, car.c), (car.r, car.c + 1)]
    } else {
        [(car.r, car.c), (car.r + 1, car.c)]
    }
}

fn board_occupied(cars: &[Car]) -> std::collections::HashSet<(usize, usize)> {
    let mut occ = std::collections::HashSet::new();
    for car in cars {
        for &cell in &car_cells(car) {
            occ.insert(cell);
        }
    }
    occ
}

fn red_car(cars: &[Car]) -> Option<Car> {
    for car in cars {
        if car.red { return Some(*car); }
    }
    None
}

fn state_key(cars: &[Car]) -> Vec<Car> {
    let mut v = cars.to_vec();
    v.sort();
    v
}

fn is_goal(cars: &[Car]) -> bool {
    match red_car(cars) {
        Some(r) => r.r == 0 && r.c == 4 && r.o == Orientation::H,
        None => false,
    }
}

fn slide_positions(car: &Car, occ: &std::collections::HashSet<(usize, usize)>) -> Vec<Car> {
    let mut pos = Vec::new();
    if car.o == Orientation::H {
        let mut k = 1;
        while car.c + k + 1 < N && !occ.contains(&(car.r, car.c + k + 1)) {
            pos.push(Car { r: car.r, c: car.c + k, o: Orientation::H, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.c >= k && !occ.contains(&(car.r, car.c - k)) {
            pos.push(Car { r: car.r, c: car.c - k, o: Orientation::H, red: car.red });
            k += 1;
        }
    } else {
        let mut k = 1;
        while car.r + k + 1 < N && !occ.contains(&(car.r + k + 1, car.c)) {
            pos.push(Car { r: car.r + k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.r >= k && !occ.contains(&(car.r - k, car.c)) {
            pos.push(Car { r: car.r - k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
    }
    pos
}

fn neighbors(cars: &[Car]) -> Vec<Vec<Car>> {
    let occ = board_occupied(cars);
    let mut out = std::collections::BTreeSet::new();
    for (i, car) in cars.iter().enumerate() {
        for newpos in slide_positions(car, &occ) {
            if newpos != *car {
                let mut newcars = cars.to_vec();
                newcars[i] = newpos;
                out.insert(state_key(&newcars));
            }
        }
    }
    out.into_iter().collect()
}

fn base_hard() -> [Car; 6] {
    [
        Car { r: 0, c: 0, o: Orientation::H, red: true },
        Car { r: 0, c: 2, o: Orientation::V, red: false },
        Car { r: 0, c: 4, o: Orientation::V, red: false },
        Car { r: 3, c: 1, o: Orientation::H, red: false },
        Car { r: 4, c: 3, o: Orientation::H, red: false },
        Car { r: 2, c: 5, o: Orientation::V, red: false },
    ]
}

fn heuristic(cars: &[Car]) -> usize {
    let r = match red_car(cars) { Some(r) => r, None => return 0 };
    if r.r != 0 { return 0; }
    let red_right = r.c + 1;
    let _occ = board_occupied(cars);
    let mut blockers: std::collections::HashSet<(usize, usize, Orientation, bool)> = std::collections::HashSet::new();
    for car in cars {
        if car.red { continue; }
        for &(a, b) in &car_cells(car) {
            if a == 0 && b > red_right {
                blockers.insert((car.r, car.c, car.o, car.red));
                break;
            }
        }
    }
    blockers.len()
}

#[derive(Clone, PartialEq, Eq, PartialOrd, Ord)]
struct HeapKey { f: usize, g: usize, key: Vec<Car> }

fn astar(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let start = state_key(start_cars);
    let mut heap: std::collections::BinaryHeap<std::cmp::Reverse<HeapKey>> = std::collections::BinaryHeap::new();
    heap.push(std::cmp::Reverse(HeapKey { f: heuristic(&start), g: 0, key: start.clone() }));
    let mut g_score: std::collections::HashMap<Vec<Car>, usize> = std::collections::HashMap::new();
    g_score.insert(start.clone(), 0);
    let mut came_from: std::collections::HashMap<Vec<Car>, Vec<Car>> = std::collections::HashMap::new();
    let mut closed: std::collections::HashSet<Vec<Car>> = std::collections::HashSet::new();
    let mut nodes = 0;
    while let Some(std::cmp::Reverse(hk)) = heap.pop() {
        if closed.contains(&hk.key) { continue; }
        closed.insert(hk.key.clone());
        nodes += 1;
        if is_goal(&hk.key) {
            let mut path = vec![hk.key.clone()];
            let mut cur = hk.key;
            while let Some(parent) = came_from.get(&cur) {
                path.push(parent.clone());
                cur = parent.clone();
            }
            path.reverse();
            return (Some(path), nodes);
        }
        for nb in neighbors(&hk.key) {
            let ng = hk.g + 1;
            let inf = usize::MAX;
            if ng < *g_score.get(&nb).unwrap_or(&inf) {
                g_score.insert(nb.clone(), ng);
                came_from.insert(nb.clone(), hk.key.clone());
                heap.push(std::cmp::Reverse(HeapKey { f: ng + heuristic(&nb), g: ng, key: nb }));
            }
        }
    }
    (None, nodes)
}

// Recherche gloutonne : suit l'heuristique (min de bloqueurs), pas d'optimalite.
fn greedy(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let mut cur = state_key(start_cars);
    let mut came_from: std::collections::HashMap<Vec<Car>, Option<Vec<Car>>> = std::collections::HashMap::new();
    came_from.insert(cur.clone(), None);
    let mut seen: std::collections::HashSet<Vec<Car>> = std::collections::HashSet::new();
    seen.insert(cur.clone());
    let mut nodes = 0;
    loop {
        nodes += 1;
        if is_goal(&cur) {
            let mut path = vec![cur.clone()];
            let mut c = cur;
            loop {
                match came_from.get(&c) {
                    Some(Some(parent)) => { path.push(parent.clone()); c = parent.clone(); }
                    _ => break,
                }
            }
            path.reverse();
            return (Some(path), nodes);
        }
        let mut best: Option<Vec<Car>> = None;
        let mut best_h = usize::MAX;
        for nb in neighbors(&cur) { // tri -> tie-break deterministe (reproductible)
            if seen.contains(&nb) { continue; }
            let h = heuristic(&nb);
            if h < best_h {
                best_h = h;
                best = Some(nb);
            }
        }
        match best {
            None => return (None, nodes), // cul-de-sac
            Some(b) => {
                seen.insert(b.clone());
                came_from.insert(b.clone(), Some(cur.clone()));
                cur = b;
            }
        }
    }
}

let instance = base_hard();
let (path_g, nodes_g) = greedy(&instance);
// recalcule astar pour le test d'optimalite (cellule autonome sous evcxr)
let (path_a, _) = astar(&instance);
let path_a = path_a.unwrap();
match path_g {
    None => println!("glouton : cul-de-sac apres {} etats (aucune solution)", nodes_g),
    Some(pg) => {
        println!("glouton : solution en {} coups, {} etats developpes", pg.len() - 1, nodes_g);
        let verdict = if pg.len() - 1 == path_a.len() - 1 { "oui" } else { "NON (sous-optimale)" };
        println!("  -> optimale? {}", verdict);
    }
};

glouton : solution en 134 coups, 135 etats developpes


  -> optimale? NON (sous-optimale)


Le contraste est la leçon : le glouton **perd l'optimalité** — sur cette instance sa solution est nettement plus longue que l'optimum (une centaine de coups contre 3), et il ne gagne pas en vitesse pour autant : il développe plus d'états que l'A*, car il suit le gradient d'heuristique au lieu de combiner coût parcouru et heuristique. L'A* est le meilleur compromis : optimal **et** économe en états.

## 6. Benchmark comparatif — le résultat mesuré

Sur un **jeu de 6 instances distinctes**, nous mesurons pour chaque algorithme le **nombre d'états développés** et la **longueur de solution**. Pour garder une comparaison à **difficulté comparable**, on **sélectionne** parmi les scrambles de la position de base les instances **non triviales** — celles où le BFS développe un volume d'états significatif (au moins 100) — et on n'en retient que 6. Les instances trop faciles sont écartées : à cette échelle, l'overhead de la structure de priorité de l'A* domine et masquerait le rôle de l'heuristique. C'est le tableau que le carnet doit produire : il rend visible, en chiffres, le rôle de l'heuristique admissible.

In [8]:
// On selectionne 6 instances non triviales (BFS developpant >= 100 etats) parmi les
// scrambles de la position de base, a difficulte comparable. Les instances trop faciles
// (peu d'etats a explorer) sont ecartees : a cette echelle l'overhead de la pile de l'A*
// domine et masque le role de l'heuristique.

const N: usize = 6;

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn car_cells(car: &Car) -> [(usize, usize); 2] {
    if car.o == Orientation::H {
        [(car.r, car.c), (car.r, car.c + 1)]
    } else {
        [(car.r, car.c), (car.r + 1, car.c)]
    }
}

fn board_occupied(cars: &[Car]) -> std::collections::HashSet<(usize, usize)> {
    let mut occ = std::collections::HashSet::new();
    for car in cars {
        for &cell in &car_cells(car) {
            occ.insert(cell);
        }
    }
    occ
}

fn red_car(cars: &[Car]) -> Option<Car> {
    for car in cars {
        if car.red { return Some(*car); }
    }
    None
}

fn state_key(cars: &[Car]) -> Vec<Car> {
    let mut v = cars.to_vec();
    v.sort();
    v
}

fn is_goal(cars: &[Car]) -> bool {
    match red_car(cars) {
        Some(r) => r.r == 0 && r.c == 4 && r.o == Orientation::H,
        None => false,
    }
}

fn slide_positions(car: &Car, occ: &std::collections::HashSet<(usize, usize)>) -> Vec<Car> {
    let mut pos = Vec::new();
    if car.o == Orientation::H {
        let mut k = 1;
        while car.c + k + 1 < N && !occ.contains(&(car.r, car.c + k + 1)) {
            pos.push(Car { r: car.r, c: car.c + k, o: Orientation::H, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.c >= k && !occ.contains(&(car.r, car.c - k)) {
            pos.push(Car { r: car.r, c: car.c - k, o: Orientation::H, red: car.red });
            k += 1;
        }
    } else {
        let mut k = 1;
        while car.r + k + 1 < N && !occ.contains(&(car.r + k + 1, car.c)) {
            pos.push(Car { r: car.r + k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.r >= k && !occ.contains(&(car.r - k, car.c)) {
            pos.push(Car { r: car.r - k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
    }
    pos
}

fn neighbors(cars: &[Car]) -> Vec<Vec<Car>> {
    let occ = board_occupied(cars);
    let mut out = std::collections::BTreeSet::new();
    for (i, car) in cars.iter().enumerate() {
        for newpos in slide_positions(car, &occ) {
            if newpos != *car {
                let mut newcars = cars.to_vec();
                newcars[i] = newpos;
                out.insert(state_key(&newcars));
            }
        }
    }
    out.into_iter().collect()
}

fn base_hard() -> [Car; 6] {
    [
        Car { r: 0, c: 0, o: Orientation::H, red: true },
        Car { r: 0, c: 2, o: Orientation::V, red: false },
        Car { r: 0, c: 4, o: Orientation::V, red: false },
        Car { r: 3, c: 1, o: Orientation::H, red: false },
        Car { r: 4, c: 3, o: Orientation::H, red: false },
        Car { r: 2, c: 5, o: Orientation::V, red: false },
    ]
}

struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn scramble(cars: &[Car], steps: usize, seed: u32) -> Vec<Car> {
    let mut rng = Mt19937::new_by_array(&[seed]);
    let mut cur = state_key(cars);
    for _ in 0..steps {
        let nb = neighbors(&cur);
        cur = rng.choice(&nb).clone();
    }
    cur
}

fn bfs(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let start = state_key(start_cars);
    let mut q: std::collections::VecDeque<Vec<Car>> = std::collections::VecDeque::new();
    q.push_back(start.clone());
    let mut came_from: std::collections::HashMap<Vec<Car>, Option<Vec<Car>>> = std::collections::HashMap::new();
    came_from.insert(start, None);
    let mut nodes = 0;
    while let Some(state) = q.pop_front() {
        nodes += 1;
        if is_goal(&state) {
            let mut path = vec![state.clone()];
            let mut cur = state;
            loop {
                match came_from.get(&cur) {
                    Some(Some(parent)) => { path.push(parent.clone()); cur = parent.clone(); }
                    _ => break,
                }
            }
            path.reverse();
            return (Some(path), nodes);
        }
        for nb in neighbors(&state) {
            if !came_from.contains_key(&nb) {
                came_from.insert(nb.clone(), Some(state.clone()));
                q.push_back(nb);
            }
        }
    }
    (None, nodes)
}

fn heuristic(cars: &[Car]) -> usize {
    let r = match red_car(cars) { Some(r) => r, None => return 0 };
    if r.r != 0 { return 0; }
    let red_right = r.c + 1;
    let _occ = board_occupied(cars);
    let mut blockers: std::collections::HashSet<(usize, usize, Orientation, bool)> = std::collections::HashSet::new();
    for car in cars {
        if car.red { continue; }
        for &(a, b) in &car_cells(car) {
            if a == 0 && b > red_right {
                blockers.insert((car.r, car.c, car.o, car.red));
                break;
            }
        }
    }
    blockers.len()
}

#[derive(Clone, PartialEq, Eq, PartialOrd, Ord)]
struct HeapKey { f: usize, g: usize, key: Vec<Car> }

fn astar(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let start = state_key(start_cars);
    let mut heap: std::collections::BinaryHeap<std::cmp::Reverse<HeapKey>> = std::collections::BinaryHeap::new();
    heap.push(std::cmp::Reverse(HeapKey { f: heuristic(&start), g: 0, key: start.clone() }));
    let mut g_score: std::collections::HashMap<Vec<Car>, usize> = std::collections::HashMap::new();
    g_score.insert(start.clone(), 0);
    let mut came_from: std::collections::HashMap<Vec<Car>, Vec<Car>> = std::collections::HashMap::new();
    let mut closed: std::collections::HashSet<Vec<Car>> = std::collections::HashSet::new();
    let mut nodes = 0;
    while let Some(std::cmp::Reverse(hk)) = heap.pop() {
        if closed.contains(&hk.key) { continue; }
        closed.insert(hk.key.clone());
        nodes += 1;
        if is_goal(&hk.key) {
            let mut path = vec![hk.key.clone()];
            let mut cur = hk.key;
            while let Some(parent) = came_from.get(&cur) {
                path.push(parent.clone());
                cur = parent.clone();
            }
            path.reverse();
            return (Some(path), nodes);
        }
        for nb in neighbors(&hk.key) {
            let ng = hk.g + 1;
            let inf = usize::MAX;
            if ng < *g_score.get(&nb).unwrap_or(&inf) {
                g_score.insert(nb.clone(), ng);
                came_from.insert(nb.clone(), hk.key.clone());
                heap.push(std::cmp::Reverse(HeapKey { f: ng + heuristic(&nb), g: ng, key: nb }));
            }
        }
    }
    (None, nodes)
}

fn greedy(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let mut cur = state_key(start_cars);
    let mut came_from: std::collections::HashMap<Vec<Car>, Option<Vec<Car>>> = std::collections::HashMap::new();
    came_from.insert(cur.clone(), None);
    let mut seen: std::collections::HashSet<Vec<Car>> = std::collections::HashSet::new();
    seen.insert(cur.clone());
    let mut nodes = 0;
    loop {
        nodes += 1;
        if is_goal(&cur) {
            let mut path = vec![cur.clone()];
            let mut c = cur;
            loop {
                match came_from.get(&c) {
                    Some(Some(parent)) => { path.push(parent.clone()); c = parent.clone(); }
                    _ => break,
                }
            }
            path.reverse();
            return (Some(path), nodes);
        }
        let mut best: Option<Vec<Car>> = None;
        let mut best_h = usize::MAX;
        for nb in neighbors(&cur) {
            if seen.contains(&nb) { continue; }
            let h = heuristic(&nb);
            if h < best_h {
                best_h = h;
                best = Some(nb);
            }
        }
        match best {
            None => return (None, nodes),
            Some(b) => {
                seen.insert(b.clone());
                came_from.insert(b.clone(), Some(cur.clone()));
                cur = b;
            }
        }
    }
}

fn bench(min_states: usize, target: usize) -> Vec<(usize, usize, usize, usize, usize, usize)> {
    let mut rows: Vec<(usize, usize, usize, usize, usize, usize)> = Vec::new();
    let mut seen: std::collections::HashSet<Vec<Car>> = std::collections::HashSet::new();
    for steps in (0..40).step_by(2) {
        for seed in 1..20 {
            let inst = scramble(&base_hard(), steps, seed);
            let key = state_key(&inst);
            if seen.contains(&key) { continue; }
            seen.insert(key);
            let (p_b, n_b) = bfs(&inst);
            if p_b.is_none() || n_b < min_states { continue; }
            let (p_a, n_a) = astar(&inst);
            let (p_g, n_g) = greedy(&inst);
            let opt = p_a.map(|p| p.len() - 1).unwrap_or(0);
            let gl = p_g.map(|p| p.len() - 1).unwrap_or(0);
            rows.push((rows.len() + 1, n_b, n_a, n_g, opt, gl));
            if rows.len() >= target { return rows; }
        }
    }
    rows
}

let rows = bench(100, 6);
println!("{:>2} | {:>10} | {:>9} | {:>12} | {:>11} | {:>14}", "#", "BFS etats", "A* etats", "glout. etats", "opt (coups)", "glout. (coups)");
println!("{}", "-".repeat(72));
for &(i, n_b, n_a, n_g, opt, gl) in &rows {
    println!("{:>2} | {:>10} | {:>9} | {:>12} | {:>11} | {:>14}", i, n_b, n_a, n_g, opt, gl);
}
let tot_b: usize = rows.iter().map(|r| r.1).sum();
let tot_a: usize = rows.iter().map(|r| r.2).sum();
println!("{}", "-".repeat(72));
println!("TOTAL etats developpes : BFS={}  A*={}  ->  A* developpe {:.1}x moins d'etats a solution optimale egale", tot_b, tot_a, tot_b as f64 / tot_a as f64);

 # |  BFS etats |  A* etats | glout. etats | opt (coups) | glout. (coups)


------------------------------------------------------------------------


 1 |        433 |       100 |          135 |           3 |            134


 2 |        620 |       262 |          135 |           3 |            134


 3 |        612 |       186 |          134 |           3 |            133


 4 |        432 |       121 |          135 |           3 |            134


 5 |        456 |       131 |          135 |           3 |            134


 6 |        432 |       121 |          135 |           3 |            134


------------------------------------------------------------------------


TOTAL etats developpes : BFS=2985  A*=921  ->  A* developpe 3.2x moins d'etats a solution optimale egale


Le tableau porte le carnet : **BFS et A* livrent la même solution optimale** (3 coups), mais **A* développe systématiquement moins d'états** — sur chacune des six instances, et en total (921 contre 2985, soit un rapport de 0,31 : le BFS développe environ 3,2 fois plus d'états). C'est le rendement mesuré de l'heuristique admissible.

Le **glouton**, lui, ne cherche pas : il livre à chaque ligne une solution **très sous-optimale** (134 coups contre 3) et son exploration est **quasi constante** — 134 ou 135 états sur les six instances, quelle que soit leur difficulté, là où l'A* varie de 100 à 262. Ce n'est pas une économie d'états, c'est la signature d'une recherche sans retour arrière : sur les deux instances les plus dures (#2 et #3), le glouton développe même *moins* d'états que l'A* (135 contre 262, 134 contre 186) et paie cette sobriété apparente de quarante-cinq fois plus de coups (134 contre 3). Suivre un gradient d'heuristique pure, au lieu de combiner coût déjà parcouru et heuristique comme le fait l'A*, c'est glisser vers la première solution venue, pas chercher. Le glouton ne gagne donc pas la course qu'il paraît gagner : il ne développe pas « plus » d'états que l'A* à chaque ligne, il en développe un nombre fixe, insensible à la difficulté — et c'est précisément ce qui le condamne.

Une note sur la mesure : les instances #4 et #6 portent cinq métriques identiques. Vérifié : leurs clés d'état sont distinctes (elles ne diffèrent que par la position d'une voiture arrière) et leurs ensembles d'états visités diffèrent — 983 communs sur 1473 — pour une taille égale. La coïncidence est donc dans la mesure, pas dans une instance dupliquée.

### 6bis. L'admissibilité, vérifiée état par état

La section 4 déclare l'heuristique **admissible** (`h(n) ≤ h*(n)` : chaque voiture bloquant la ligne 0 à droite de la rouge doit être évacuée par **au moins un coup distinct**, donc le nombre de bloqueurs borne le coût restant). La section 6 en vérifie la **conséquence** — l'A* retrouve la solution optimale sur 6 instances. Mais **aucune cellule ne mesure l'inégalité elle-même**, `h(n) ≤ h*(n)`, état par état.

La section 6bis le fait : sur un balayage d'états brouillés, le **BFS du carnet** (indépendant de l'heuristique) fournit `h*(n)` = longueur de la solution optimale, et on compare `h(n)` à `h*(n)` sur chaque état.

In [9]:
const N: usize = 6;

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn car_cells(car: &Car) -> [(usize, usize); 2] {
    if car.o == Orientation::H {
        [(car.r, car.c), (car.r, car.c + 1)]
    } else {
        [(car.r, car.c), (car.r + 1, car.c)]
    }
}

fn board_occupied(cars: &[Car]) -> std::collections::HashSet<(usize, usize)> {
    let mut occ = std::collections::HashSet::new();
    for car in cars {
        for &cell in &car_cells(car) {
            occ.insert(cell);
        }
    }
    occ
}

fn red_car(cars: &[Car]) -> Option<Car> {
    for car in cars {
        if car.red { return Some(*car); }
    }
    None
}

fn state_key(cars: &[Car]) -> Vec<Car> {
    let mut v = cars.to_vec();
    v.sort();
    v
}

fn is_goal(cars: &[Car]) -> bool {
    match red_car(cars) {
        Some(r) => r.r == 0 && r.c == 4 && r.o == Orientation::H,
        None => false,
    }
}

fn slide_positions(car: &Car, occ: &std::collections::HashSet<(usize, usize)>) -> Vec<Car> {
    let mut pos = Vec::new();
    if car.o == Orientation::H {
        let mut k = 1;
        while car.c + k + 1 < N && !occ.contains(&(car.r, car.c + k + 1)) {
            pos.push(Car { r: car.r, c: car.c + k, o: Orientation::H, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.c >= k && !occ.contains(&(car.r, car.c - k)) {
            pos.push(Car { r: car.r, c: car.c - k, o: Orientation::H, red: car.red });
            k += 1;
        }
    } else {
        let mut k = 1;
        while car.r + k + 1 < N && !occ.contains(&(car.r + k + 1, car.c)) {
            pos.push(Car { r: car.r + k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
        let mut k = 1;
        while car.r >= k && !occ.contains(&(car.r - k, car.c)) {
            pos.push(Car { r: car.r - k, c: car.c, o: Orientation::V, red: car.red });
            k += 1;
        }
    }
    pos
}

fn neighbors(cars: &[Car]) -> Vec<Vec<Car>> {
    let occ = board_occupied(cars);
    let mut out = std::collections::BTreeSet::new();
    for (i, car) in cars.iter().enumerate() {
        for newpos in slide_positions(car, &occ) {
            if newpos != *car {
                let mut newcars = cars.to_vec();
                newcars[i] = newpos;
                out.insert(state_key(&newcars));
            }
        }
    }
    out.into_iter().collect()
}

fn base_hard() -> [Car; 6] {
    [
        Car { r: 0, c: 0, o: Orientation::H, red: true },
        Car { r: 0, c: 2, o: Orientation::V, red: false },
        Car { r: 0, c: 4, o: Orientation::V, red: false },
        Car { r: 3, c: 1, o: Orientation::H, red: false },
        Car { r: 4, c: 3, o: Orientation::H, red: false },
        Car { r: 2, c: 5, o: Orientation::V, red: false },
    ]
}

struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn scramble(cars: &[Car], steps: usize, seed: u32) -> Vec<Car> {
    let mut rng = Mt19937::new_by_array(&[seed]);
    let mut cur = state_key(cars);
    for _ in 0..steps {
        let nb = neighbors(&cur);
        cur = rng.choice(&nb).clone();
    }
    cur
}

fn bfs(start_cars: &[Car]) -> (Option<Vec<Vec<Car>>>, usize) {
    let start = state_key(start_cars);
    let mut q: std::collections::VecDeque<Vec<Car>> = std::collections::VecDeque::new();
    q.push_back(start.clone());
    let mut came_from: std::collections::HashMap<Vec<Car>, Option<Vec<Car>>> = std::collections::HashMap::new();
    came_from.insert(start, None);
    let mut nodes = 0;
    while let Some(state) = q.pop_front() {
        nodes += 1;
        if is_goal(&state) {
            let mut path = vec![state.clone()];
            let mut cur = state;
            loop {
                match came_from.get(&cur) {
                    Some(Some(parent)) => { path.push(parent.clone()); cur = parent.clone(); }
                    _ => break,
                }
            }
            path.reverse();
            return (Some(path), nodes);
        }
        for nb in neighbors(&state) {
            if !came_from.contains_key(&nb) {
                came_from.insert(nb.clone(), Some(state.clone()));
                q.push_back(nb);
            }
        }
    }
    (None, nodes)
}

fn heuristic(cars: &[Car]) -> usize {
    let r = match red_car(cars) { Some(r) => r, None => return 0 };
    if r.r != 0 { return 0; }
    let red_right = r.c + 1;
    let _occ = board_occupied(cars);
    let mut blockers: std::collections::HashSet<(usize, usize, Orientation, bool)> = std::collections::HashSet::new();
    for car in cars {
        if car.red { continue; }
        for &(a, b) in &car_cells(car) {
            if a == 0 && b > red_right {
                blockers.insert((car.r, car.c, car.o, car.red));
                break;
            }
        }
    }
    blockers.len()
}

let t0 = std::time::Instant::now();
let mut n_checked = 0;
let mut viol = 0;
let mut rows21: Vec<(usize, usize, usize, usize, &str)> = Vec::new();
let mut seen: std::collections::HashSet<Vec<Car>> = std::collections::HashSet::new();

// Balayage : brouillages 0..12 coups (pas 2) x 7 graines, dedupes.
// Le cap de 30 etats sature le balayage avant steps=12 : prof max mesuree = 10.
// h*(n) = len(path) - 1 par le BFS du carnet (reference independante de l'heuristique).
'graines: for steps in (0..13).step_by(2) {
    for seed in 1..8 {
        let inst = scramble(&base_hard(), steps, seed);
        let key = state_key(&inst);
        if seen.contains(&key) { continue; } // etat deja mesure (brouillages qui reviennent au meme etat)
        seen.insert(key);
        let (p, _) = bfs(&inst);
        let p = match p { Some(p) => p, None => continue }; // etat non resoluble : hors du balayage
        let hstar = p.len() - 1; // le vrai cout optimal
        let h = heuristic(&inst);
        n_checked += 1;
        let ok = h <= hstar;
        if !ok { viol += 1; }
        rows21.push((steps, seed as usize, hstar, h, if ok { "OUI" } else { "NON" }));
        if n_checked >= 30 { break; }
    }
    if n_checked >= 30 { break 'graines; }
}

let elapsed = t0.elapsed().as_secs_f64();

println!("{:>4} | {:>6} | {:>8} | {:>9} | h <= h*", "prof", "graine", "h* (BFS)", "h (heur.)");
println!("{}", "-".repeat(46));
for &(steps, seed, hstar, h, ref flag) in &rows21 {
    println!("{:>4} | {:>6} | {:>8} | {:>9} | {}", steps, seed, hstar, h, flag);
}
println!("{}", "-".repeat(46));
println!("{} etats mesures (h* par le BFS du carnet) en {:.1} s :", n_checked, elapsed);
println!("  heuristique <= h* : {}/{}", n_checked - viol, n_checked);
println!("Violations de l'admissibilite : {}.", viol);

prof | graine | h* (BFS) | h (heur.) | h <= h*


----------------------------------------------


   0 |      1 |        3 |         2 | OUI


   2 |      1 |        3 |         2 | OUI


   2 |      2 |        3 |         2 | OUI


   2 |      3 |        3 |         2 | OUI


... (26 outputs left out)

----------------------------------------------


30 etats mesures (h* par le BFS du carnet) en 0.1 s :


  heuristique <= h* : 30/30


Violations de l'admissibilite : 0.


**Lecture du balayage.** Sur **30 états** brouillés, `h(n) ≤ h*(n)` tient **partout** : **0 violation**. Le `h*(n)` est fourni par le **BFS du carnet** — une référence **indépendante** de l'heuristique — donc la borne est comparée à un coût optimal réel, pas à elle-même.

Le balayage est **non dégénéré** : plusieurs états sont à **borne stricte** (`h* = 4, h = 3`, `h* = 3, h = 2`…), et un seul est à l'égalité — l'état-but lui-même (`h* = 0, h = 0`). Le minorant n'est donc pas un `h = h*` partout.

**Portée honnête.** La vérification porte sur **cette instance** (grille 6×6, 6 voitures) et des états **faibles** (brouillage ≤ 10 coups — le cap de 30 états du balayage sature avant steps=12 —, `h* ≤ 4`, là où le BFS du carnet reste exécutable en quelques dixièmes de seconde). L'argument d'admissibilité est général (chaque bloqueur coûte un coup distinct) ; ce balayage en est le **témoin falsifiable** — une seule ligne `h > h*` y invaliderait l'implémentation de `heuristic`. Non transférable à d'autres instances.

## 7. Exercices

Trois exercices, un par mécanisme du carnet : le test de but, l'heuristique admissible, le voisinage.

**Politique de sortie.** Les cellules d'exercice ne produisent **aucune sortie** à l'exécution, et c'est voulu : ce sont des `fn`, et une définition de fonction n'affiche rien. Le `println!("Exercice a completer")` qu'elles contiennent vit dans le corps de la fonction et ne se joue que lorsqu'on l'appelle. Complétez la fonction, puis appelez-la sur une instance — `is_goal_exo(&base_hard())` par exemple — pour voir votre résultat : c'est cette sortie-là qui compte, pas celle du carnet.

### Exercice 1 — Le test de but

Complète `is_goal_exo` : renvoyer `true` si la voiture rouge est bien à la sortie (horizontale, ligne 0, case gauche en colonne 4).

**Indice** : récupère la voiture rouge avec `red_car`, puis vérifie ses trois coordonnées `(ligne, col, orientation)`.

In [10]:
#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn is_goal_exo(_cars: &[Car]) -> bool {
    // TODO etudiant
    println!("Exercice a completer");
    false // TODO: bool
}

### Exercice 2 — L'heuristique admissible

Réimplimente `heuristic_exo` : compter les voitures (hors la rouge) qui occupent la **ligne 0 à droite de la case droite de la rouge**.

**Indice** : détermine la colonne `red_right` de la case droite de la rouge, puis compte les voitures distinctes ayant au moins une cellule en `(0, c)` avec `c > red_right`.

In [11]:
#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn heuristic_exo(_cars: &[Car]) -> usize {
    // TODO etudiant
    println!("Exercice a completer");
    0 // TODO: int
}

### Exercice 3 — Un voisinage

Complète `slide_exo` : pour une voiture donnée et un ensemble `occ` de cellules occupées, renvoyer la **liste des positions** atteignables par un glissement (les deux sens de l'axe).

**Indice** : pour un sens, incrémente un pas `k` tant que la cellule de tête du glissement est dans les bornes et absente de `occ` ; ajoute la nouvelle position à chaque pas valide.

In [12]:
#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
enum Orientation { H, V }

#[derive(Clone, Copy, PartialEq, Eq, PartialOrd, Ord, Hash, Debug)]
struct Car { r: usize, c: usize, o: Orientation, red: bool }

fn slide_exo(_car: &Car, _occ: &std::collections::HashSet<(usize, usize)>) -> Vec<Car> {
    // TODO etudiant
    println!("Exercice a completer");
    Vec::new() // TODO: list[tuple]
}

## Conclusion

Le Rush Hour est le terrain d'essai idéal du **triangle de la recherche** : le **BFS** est optimal mais dépense ; l'**A\*** avec une **heuristique admissible** (les bloqueurs de la rouge) est **à la fois optimal et économe** — c'est la leçon centrale, mesurée sur un jeu de 6 instances (921 états contre 2985, soit 3,2 fois moins d'états développés) ; la **recherche gloutonne** explore un nombre d'états quasi constant (134-135, quelle que soit l'instance) mais **sacrifie l'optimalité** (134 coups contre 3) : elle ne cherche pas, elle glisse. Choisir une heuristique, c'est choisir où tomber dans ce triangle : l'admissibilité est la propriété qui permet de ne pas sacrifier l'optimalité. Ce carnet complète, avec le 15-puzzle de la Partie 1, la famille des puzzles à glissement de la série et ferme la boucle sur les heuristiques admissibles introduites dans les Parties 1-2.

> Carnet issu de l'EPIC #26534.